# Bitcoin / LightGBM

**Historical exploratory experiment — not a validated benchmark.**

Source workspace filename: `Bitcoin.ipynb`. See `../README.md` and `../../docs/source_manifest.json` for status and provenance.

Saved outputs were cleared for publication. Dataset paths were made relative; run from `notebooks/exploratory/`. Model dependencies and data access must be configured separately. Do not quote accuracy from this notebook without reproducing its split, target alignment, and calibration protocol.


In [ ]:
############# Bitcoin Lowest Price Forecasting với Sktime + LightGBM

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["PYTHONWARNINGS"] = "ignore"

# LightGBM imports
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error
from pandas.tseries.offsets import BDay
import matplotlib.pyplot as plt

# Binance API
import requests

# Sktime imports - POWERFUL unified time series ML library
try:
    from sktime.forecasting.compose import make_reduction
    from sktime.forecasting.model_selection import temporal_train_test_split
    from sktime.performance_metrics.forecasting import mean_absolute_percentage_error as mape_sktime
    print("[INFO] ✅ Sktime installed")
except ImportError:
    print("[WARNING] Sktime not installed. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'sktime'])
    from sktime.forecasting.compose import make_reduction
    from sktime.forecasting.model_selection import temporal_train_test_split
    from sktime.performance_metrics.forecasting import mean_absolute_percentage_error as mape_sktime
    print("[INFO] ✅ Sktime installed successfully")

print("[INFO] LightGBM + Sktime Forecasting Environment Ready")

In [ ]:
# ========== LẤY DỮ LIỆU BITCOIN TỪ BINANCE API (TOÀN BỘ LỊCH SỬ) ==========
def fetch_bitcoin_historical_data(symbol='BTCUSDT', interval='1d', start_date='2017-08-17'):
    """
    Lấy toàn bộ dữ liệu Bitcoin từ Binance API (từ khi niêm yết)
    
    Parameters:
    - symbol: Trading pair (default: BTCUSDT)
    - interval: Timeframe (1m, 5m, 15m, 1h, 4h, 1d, 1w, 1M)
    - start_date: Ngày bắt đầu (Binance niêm yết BTCUSDT từ 17/08/2017)
    
    Returns:
    - DataFrame với columns: ds (datetime), y (Low price), volume
    """
    try:
        print(f"[INFO] Đang lấy TOÀN BỘ dữ liệu {symbol} từ Binance API...")
        print(f"[INFO] Từ ngày: {start_date} đến hiện tại")
        
        # Binance API endpoint
        url = "https://api.binance.com/api/v3/klines"
        
        # Convert start_date to timestamp
        start_timestamp = int(pd.Timestamp(start_date).timestamp() * 1000)
        
        all_data = []
        current_timestamp = start_timestamp
        
        # Binance giới hạn 1000 nến/request, cần loop để lấy hết
        batch_count = 0
        
        while True:
            batch_count += 1
            print(f"[INFO] Đang lấy batch {batch_count}...", end='\r')
            
            params = {
                'symbol': symbol,
                'interval': interval,
                'startTime': current_timestamp,
                'limit': 1000
            }
            
            response = requests.get(url, params=params)
            response.raise_for_status()
            
            data = response.json()
            
            if not data:
                break
            
            all_data.extend(data)
            
            # Update timestamp cho batch tiếp theo
            # Lấy close_time của nến cuối cùng + 1ms
            current_timestamp = data[-1][6] + 1
            
            # Nếu đã đến thời điểm hiện tại, dừng lại
            if current_timestamp >= int(datetime.now().timestamp() * 1000):
                break
            
            # Rate limiting - tránh bị ban
            import time
            time.sleep(0.1)
        
        print(f"\n[INFO] ✅ Đã lấy {batch_count} batches, tổng {len(all_data)} nến")
        
        # Parse data
        # Binance klines format:
        # [Open time, Open, High, Low, Close, Volume, Close time, ...]
        df = pd.DataFrame(all_data, columns=[
            'open_time', 'open', 'high', 'low', 'close', 'volume',
            'close_time', 'quote_volume', 'trades', 'taker_buy_base',
            'taker_buy_quote', 'ignore'
        ])
        
        # Convert to proper types
        df['ds'] = pd.to_datetime(df['open_time'], unit='ms')
        df['y'] = pd.to_numeric(df['low'], errors='coerce')  # Low price
        df['volume'] = pd.to_numeric(df['volume'], errors='coerce')
        
        # Keep only needed columns
        df = df[['ds', 'y', 'volume']].copy()
        
        # Clean data
        df = df.dropna(subset=['ds', 'y', 'volume'])
        df = df[df['y'] > 0]
        df = df.sort_values('ds').reset_index(drop=True)
        
        # Remove duplicates (nếu có)
        df = df.drop_duplicates(subset=['ds'], keep='last')
        
        print(f"[INFO] ✅ Sau khi clean: {len(df)} nến dữ liệu")
        print(f"[INFO] Khoảng thời gian: {df['ds'].min().date()} → {df['ds'].max().date()}")
        print(f"[INFO] Tổng số ngày: {(df['ds'].max() - df['ds'].min()).days} ngày")
        return df
        
    except Exception as e:
        print(f"[ERROR] Lỗi khi lấy dữ liệu từ Binance: {e}")
        raise

# Lấy dữ liệu Bitcoin từ khi Binance niêm yết (17/08/2017)
df = fetch_bitcoin_historical_data(symbol='BTCUSDT', interval='1d', start_date='2017-08-17')

print(f"\n{'='*70}")
print(f"📊 THỐNG KÊ DỮ LIỆU BITCOIN")
print(f"{'='*70}")
print(f"[INFO] Nguồn: Binance API (BTCUSDT)")
print(f"[INFO] Số nến: {len(df)}")
print(f"[INFO] Từ: {df['ds'].min().date()} → {df['ds'].max().date()}")
print(f"[INFO] Tổng số năm: {(df['ds'].max() - df['ds'].min()).days / 365.25:.1f} năm")
print(f"\n[INFO] Giá Low:")
print(f"   - Min: ${df['y'].min():.2f}")
print(f"   - Max: ${df['y'].max():.2f}")
print(f"   - Mean: ${df['y'].mean():.2f}")
print(f"   - Median: ${df['y'].median():.2f}")
print(f"\n[INFO] Volume:")
print(f"   - Min: {df['volume'].min():.2f} BTC")
print(f"   - Max: {df['volume'].max():.2f} BTC")
print(f"   - Mean: {df['volume'].mean():.2f} BTC")
print(f"{'='*70}")

df.head(10)

In [ ]:
print(len(df))

In [ ]:
df.dropna()

In [ ]:
df.fillna(method='ffill', inplace=True)

In [ ]:
# ========== TRAIN LIGHTGBM MODEL WITH SKTIME ==========

# Chuẩn bị dữ liệu cho sktime
print("[INFO] Chuẩn bị dữ liệu cho Sktime...")
df_ts = df.copy()
df_ts = df_ts.set_index('ds')
df_ts = df_ts.sort_index()

# QUAN TRỌNG: Sktime yêu cầu index có frequency
# Với Bitcoin giao dịch 24/7, dùng daily frequency
print("[INFO] Setting daily frequency...")

# Set frequency to daily
df_ts = df_ts.asfreq('D', method='ffill')

# Bây giờ index đã có freq='D' tự động
y = df_ts['y']
X = df_ts[['volume']]  # Exogenous variables

print(f"[INFO] Time series length: {len(y)}")
print(f"[INFO] Date range: {y.index.min().date()} → {y.index.max().date()}")
print(f"[INFO] Index frequency: {y.index.freq}")

# Tạo LightGBM regressor với tuned parameters
lgbm_regressor = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=10,
    num_leaves=127,
    min_child_samples=5,
    min_child_weight=0.001,
    objective='mae',
    boosting_type='gbdt',
    subsample=0.9,
    subsample_freq=3,
    colsample_bytree=0.9,
    reg_alpha=0.05,
    reg_lambda=0.5,
    min_split_gain=0.001,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

# Sktime's make_reduction: Converts regressor to forecaster
# strategy='recursive': Multi-step forecasting (như iterative approach)
# window_length: Số lags để dùng (tương đương lag_1 ... lag_150)
print("\n[INFO] Đang train Sktime + LightGBM model...")
print("[INFO] Sử dụng recursive strategy với 150 lags window")

forecaster = make_reduction(
    lgbm_regressor,
    strategy="recursive",  # Recursive multi-step forecasting
    window_length=150,     # Use 150 past values as features
    scitype="tabular-regressor"
)

# Train model
print("[INFO] Training forecaster...")
forecaster.fit(y=y, X=X)

print("[INFO] ✅ Training hoàn tất!")
print(f"[INFO] Model type: {type(forecaster).__name__}")
print(f"[INFO] Strategy: Recursive multi-step forecasting")
print(f"[INFO] Window length: 150 lags")

In [ ]:
# ========== DỰ ĐOÁN 60 NGÀY VỚI SKTIME ==========

print("[INFO] Dự đoán 60 ngày tiếp theo với Sktime...")

# Tạo exogenous variables cho future (60 days)
recent_volume = X['volume'].tail(30).mean()

# Tạo future dates với daily frequency (Bitcoin trades 24/7)
last_date = y.index.max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=60, freq='D')

X_future = pd.DataFrame({
    'volume': [recent_volume] * 60
}, index=future_dates)

# Dự đoán với sktime - Sử dụng relative horizon
print("[INFO] Running forecast...")
forecast_60d = forecaster.predict(fh=list(range(1, 61)), X=X_future)

# Convert to dataframe
forecast_df = pd.DataFrame({
    'ds': forecast_60d.index,
    'yhat1': forecast_60d.values
})

print(f"[INFO] ✅ Dự đoán hoàn tất!")
print(f"\n📊 Dự đoán 60 ngày đầu tiên:")
print(forecast_df.head(10))
print(f"\n📊 Dự đoán 60 ngày cuối:")
print(forecast_df.tail(5))
print(f"\nDự đoán trung bình: ${forecast_df['yhat1'].mean():.2f}")
print(f"Dự đoán min: ${forecast_df['yhat1'].min():.2f}")
print(f"Dự đoán max: ${forecast_df['yhat1'].max():.2f}")

In [ ]:
# ========== VẼ BIỂU ĐỒ FORECAST ==========
fig, ax = plt.subplots(figsize=(14, 6))

# Plot historical data
n_lags = 200
y_plot = y.tail(n_lags)
ax.plot(y_plot.index, y_plot.values, label='Actual', marker='.', linewidth=2, color='black')

# Plot forecast
ax.plot(forecast_df['ds'], forecast_df['yhat1'], label='Forecast (60 days - Sktime)', 
        marker='.', linewidth=2, color='#E63946', linestyle='--')

ax.set_title('Bitcoin Low Price Forecast - Sktime + LightGBM\n(Last 200 Days + 60 Days Forecast)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (USD)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n📈 Forecast Statistics:")
print(f"   Mean: ${forecast_df['yhat1'].mean():.2f}")
print(f"   Std: ${forecast_df['yhat1'].std():.2f}")
print(f"   Trend: {((forecast_df['yhat1'].iloc[-1] / forecast_df['yhat1'].iloc[0]) - 1) * 100:.2f}%")

In [ ]:
# ========== TEST MAPE 30 NGÀY VỚI SKTIME ==========

print("=" * 70)
print("TEST MAPE 30 NGÀY - SKTIME + LIGHTGBM (BITCOIN)")
print("=" * 70)

# Sktime temporal train-test split
test_size = 30

# Split data - y và X đã có freq từ cell trước (asfreq tự động set)
y_train, y_test = temporal_train_test_split(y, test_size=test_size)
X_train = X.loc[y_train.index]
X_test = X.loc[y_test.index]

print(f"[INFO] Train size: {len(y_train)} days (đến {y_train.index.max().date()})")
print(f"[INFO] Test size: {len(y_test)} days (từ {y_test.index.min().date()} đến {y_test.index.max().date()})")
print(f"[INFO] Train freq: {y_train.index.freq}, Test freq: {y_test.index.freq}")

# Retrain forecaster trên training data
forecaster_test = make_reduction(
    LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.02,
        max_depth=10,
        num_leaves=127,
        min_child_samples=5,
        min_child_weight=0.001,
        objective='mae',
        boosting_type='gbdt',
        subsample=0.9,
        subsample_freq=3,
        colsample_bytree=0.9,
        reg_alpha=0.05,
        reg_lambda=0.5,
        min_split_gain=0.001,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    ),
    strategy="recursive",
    window_length=150,
    scitype="tabular-regressor"
)

print("\n[INFO] Training model trên dữ liệu train...")
forecaster_test.fit(y=y_train, X=X_train)

# Tạo forecast horizon cho test - use relative horizon
print("[INFO] Dự đoán 30 ngày test...")
y_pred = forecaster_test.predict(fh=list(range(1, test_size + 1)), X=X_test)

# So sánh với actual
comparison_df = pd.DataFrame({
    'ds': y_test.index,
    'y': y_test.values,
    'yhat': y_pred.values
})

# Tính metrics
mape_score = mean_absolute_percentage_error(comparison_df['y'], comparison_df['yhat']) * 100
mae = mean_absolute_error(comparison_df['y'], comparison_df['yhat'])
rmse = np.sqrt(mean_squared_error(comparison_df['y'], comparison_df['yhat']))

print(f"\n{'='*70}")
print("📊 KẾT QUẢ TEST MAPE 30 NGÀY (BITCOIN):")
print(f"{'='*70}")
print(f"✅ MAPE: {mape_score:.2f}%")
print(f"✅ MAE: ${mae:.2f}")
print(f"✅ RMSE: ${rmse:.2f}")
print(f"✅ Số ngày test: {len(comparison_df)}")
print(f"✅ Trung bình Actual: ${comparison_df['y'].mean():.2f}")
print(f"✅ Trung bình Predicted: ${comparison_df['yhat'].mean():.2f}")

# Sai số chi tiết
comparison_df['error'] = np.abs(comparison_df['y'] - comparison_df['yhat'])
comparison_df['error_pct'] = (comparison_df['error'] / comparison_df['y']) * 100

print("\n[DETAIL] Top 5 ngày có sai số cao nhất:")
top_errors = comparison_df.nlargest(5, 'error_pct')[['ds', 'y', 'yhat', 'error', 'error_pct']]
top_errors_display = top_errors.copy()
top_errors_display['ds'] = pd.to_datetime(top_errors_display['ds']).dt.date
print(top_errors_display.to_string(index=False))
print(f"{'='*70}")

In [ ]:
# ========== HIỂN THỊ KẾT QUẢ MAPE VỚI SKTIME ==========
if 'mape_score' in locals() and mape_score is not None and 'comparison_df' in locals():
    print("=" * 70)
    print("📊 PHÂN TÍCH CHI TIẾT - SKTIME + LIGHTGBM (BITCOIN)")
    print("=" * 70)
    print(f"\n✅ MAPE: {mape_score:.2f}%")
    print(f"✅ MAE: ${mae:.2f}")
    print(f"✅ RMSE: ${rmse:.2f}")
    print(f"✅ Số ngày test: {len(comparison_df)}")

    # Vẽ biểu đồ so sánh
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    # Plot 1: So sánh giá trị
    ax1 = axes[0]
    ax1.plot(comparison_df['ds'], comparison_df['y'], label='Actual', 
             marker='o', linewidth=2, markersize=6, color='#457B9D')
    ax1.plot(comparison_df['ds'], comparison_df['yhat'], label='Predicted (Sktime)', 
             marker='x', linewidth=2, markersize=6, color='#E63946')
    ax1.set_title(f'Bitcoin - Sktime + LightGBM: Actual vs Predicted (MAPE: {mape_score:.2f}%)', 
                  fontsize=14, fontweight='bold')
    ax1.set_xlabel('Date', fontsize=11)
    ax1.set_ylabel('Bitcoin Low Price (USD)', fontsize=11)
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Sai số theo ngày
    ax2 = axes[1]
    colors = ['#06A77D' if e < 5 else '#F18F01' if e < 8 else '#C73E1D' 
              for e in comparison_df['error_pct']]
    ax2.bar(range(len(comparison_df)), comparison_df['error_pct'], color=colors, alpha=0.8)
    ax2.set_title('Error Percentage by Day (Green: <5%, Orange: 5-8%, Red: >8%)', 
                  fontsize=12, fontweight='bold')
    ax2.set_xlabel('Day', fontsize=11)
    ax2.set_ylabel('Error %', fontsize=11)
    ax2.axhline(y=5, color='green', linestyle='--', alpha=0.5, linewidth=2, label='5% threshold')
    ax2.axhline(y=mape_score, color='blue', linestyle=':', alpha=0.7, linewidth=2, label=f'Mean: {mape_score:.2f}%')
    ax2.legend(loc='best', fontsize=9)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Cumulative error
    ax3 = axes[2]
    comparison_df['cumulative_error'] = comparison_df['error'].cumsum()
    ax3.plot(range(len(comparison_df)), comparison_df['cumulative_error'], 
             color='#8338EC', linewidth=2, marker='.')
    ax3.set_title('Cumulative Absolute Error Over Time', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Day', fontsize=11)
    ax3.set_ylabel('Cumulative Error (USD)', fontsize=11)
    ax3.grid(True, alpha=0.3)
    ax3.fill_between(range(len(comparison_df)), comparison_df['cumulative_error'], alpha=0.3, color='#8338EC')
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Thống kê chi tiết
    print("\n" + "="*70)
    print("📈 THỐNG KÊ SAI SỐ:")
    print("="*70)
    print(f"Mean Error %: {comparison_df['error_pct'].mean():.2f}%")
    print(f"Median Error %: {comparison_df['error_pct'].median():.2f}%")
    print(f"Std Error %: {comparison_df['error_pct'].std():.2f}%")
    print(f"Min Error %: {comparison_df['error_pct'].min():.2f}%")
    print(f"Max Error %: {comparison_df['error_pct'].max():.2f}%")
    print(f"\n🎯 Ngày có error < 5%: {(comparison_df['error_pct'] < 5).sum()} / {len(comparison_df)} ({(comparison_df['error_pct'] < 5).sum() / len(comparison_df) * 100:.1f}%)")
    print(f"🎯 Ngày có error < 8%: {(comparison_df['error_pct'] < 8).sum()} / {len(comparison_df)} ({(comparison_df['error_pct'] < 8).sum() / len(comparison_df) * 100:.1f}%)")
    
else:
    print("⚠️ Chưa có kết quả MAPE. Hãy chạy cell test MAPE trước.")

## 🚀 Sktime + LightGBM - Bitcoin Forecasting

### ✅ Ưu điểm của Sktime:
- **Unified API**: Giống sklearn, dễ học và sử dụng
- **Multiple strategies**: Recursive, direct, multioutput forecasting
- **make_reduction**: Convert bất kỳ sklearn/lightgbm regressor thành forecaster
- **Proper time series validation**: `temporal_train_test_split`, `SlidingWindowSplitter`
- **Rich ecosystem**: Kết hợp dễ dàng với sklearn, statsmodels, prophet
- **Production-ready**: Clean API, well-tested, maintained by community

### 🔥 Sktime Strategies:
```python
# Recursive: Dùng predictions làm input cho step tiếp theo
strategy="recursive"  # ← Đang dùng

# Direct: Train riêng model cho mỗi horizon
strategy="direct"

# MultiOutput: Predict multiple steps cùng lúc
strategy="multioutput"
```

### 💡 Kết hợp LightGBM + Sktime cho Bitcoin:
- **LightGBM**: Fast gradient boosting, handles non-linear patterns
- **Sktime**: Professional time series framework với sklearn API
- **make_reduction**: Tự động convert regressor → forecaster
- **Binance API**: Real-time data, 24/7 trading, high liquidity
- **Result**: Enterprise-grade crypto forecasting với minimal code!

### 🎯 Sktime Features:
| Feature | Description |
|---------|-------------|
| `make_reduction` | Convert any regressor to forecaster |
| `temporal_train_test_split` | Proper time series split |
| `ForecastingHorizon` | Flexible horizon specification |
| `ColumnEnsembleForecaster` | Ensemble multiple models |
| `TransformedTargetForecaster` | Add transformations |

### 📚 Next Steps:
1. Thử `strategy="direct"` để so sánh với recursive
2. Use `ReducedForecaster` với multiple windows
3. Ensemble với ARIMA: `ColumnEnsembleForecaster`
4. Add feature engineering với `Detrender`, `Deseasonalizer`
5. Hyperparameter tuning với `ForecastingGridSearchCV`
6. Thử với các altcoins khác (ETH, BNB, SOL, etc.)